<a href="https://colab.research.google.com/github/Peeyusj/rag_sratch/blob/main/week23_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install chromadb sentence-transformers groq langchain-text-splitters --quiet

In [2]:
import urllib.request

url = "https://www.gutenberg.org/cache/epub/19630/pg19630.txt"

with urllib.request.urlopen(url) as response:
    raw_text = response.read().decode('utf-8')

# Strip to just the main content
clean_text = raw_text[2275:284556]

print(len(clean_text))
print(clean_text[:200])

282281
BOOK I

ASTRA DARSANA

(The Tournament)


The scene of the Epic is the ancient kingdom of the Kurus which
flourished along the upper course of the Ganges; and the historical
fact on which the


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ".", " "],
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_text(clean_text)
print(f"Total chunks: {len(chunks)}")
print(chunks[0])

Total chunks: 610
BOOK I

ASTRA DARSANA

(The Tournament)


The scene of the Epic is the ancient kingdom of the Kurus which
flourished along the upper course of the Ganges; and the historical
fact on which the Epic is based is a great war which took place
between the Kurus and a neighbouring tribe, the Panchalas, in the
thirteenth or fourteenth century before Christ.

According to the Epic, Pandu and Dhrita-rashtra, who was born blind,


In [6]:
from google.colab import drive
drive.mount('/content/drive')

import chromadb
from sentence_transformers import SentenceTransformer

# Save ChromaDB to Google Drive — survives session restarts
client = chromadb.PersistentClient(path="/content/drive/MyDrive/chroma_db")
collection = client.get_or_create_collection("mahabharata")

# Always load the model — needed for both storing AND querying
model = SentenceTransformer('all-MiniLM-L6-v2')

if collection.count() == 0:
    print("Embedding chunks and storing in ChromaDB...")
    embeddings = model.encode(chunks).tolist()

    collection.add(
        ids=[f"chunk_{i}" for i in range(len(chunks))],
        embeddings=embeddings,
        documents=chunks
    )
    print(f"Stored {collection.count()} chunks")
else:
    print(f"Collection already exists with {collection.count()} chunks — skipping embedding")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Collection already exists with 610 chunks — skipping embedding


In [7]:
from groq import Groq
from google.colab import userdata

groq_client = Groq(api_key=userdata.get('GROQ_API_KEY'))

def ask_mahabharata(question):
    # Step 1: Embed the question
    question_embedding = model.encode([question]).tolist()

    # Step 2: Retrieve top 3 relevant chunks from ChromaDB
    results = collection.query(
        query_embeddings=question_embedding,
        n_results=3
    )

    # Step 3: Join chunks into context string
    context = "\n\n".join(results['documents'][0])

    # Step 4: Send context + question to Groq
    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": "You are an expert on the Mahabharata. Answer questions using only the provided context."},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}
        ]
    )

    return response.choices[0].message.content

# Test it
answer = ask_mahabharata("Who was Arjuna's greatest enemy?")
print(answer)

Based on the given context, Arjuna's greatest opponent is Duryodhan. It is mentioned that Duryodhan "vainly" hurled his lance against Arjuna's face, suggesting a fierce battle between the two.
